In [5]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score

In [6]:
df = pd.read_csv('hotel_bookings_course_release_v1.csv')

print("--- Dimensões do dataset ---")
print(f"Total de reservas: {df.shape[0]}")
print(f"Total de atributos: {df.shape[1]}\n")

--- Dimensões do dataset ---
Total de reservas: 119390
Total de atributos: 32



In [7]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

print("--- Valores Nulos Numéricos ---")
nulos_num = df[num_cols].isnull().sum()
print(nulos_num[nulos_num > 0] if nulos_num.sum() > 0 else "Sem nulo")

print("\n--- Valores Nulos de Categoria ---")
nulos_cat = df[cat_cols].isnull().sum()
print(nulos_cat[nulos_cat > 0] if nulos_cat.sum() > 0 else "Sem nulo")

print("\n--- Estatisticas de Deteção de Anomalias ---")
print(df[num_cols].describe().T[['min', 'max', 'mean', '50%']])

--- Valores Nulos Numéricos ---
children         4
agent        16340
company     112593
dtype: int64

--- Valores Nulos de Categoria ---
country    488
dtype: int64

--- Estatisticas de Deteção de Anomalias ---
                                    min     max         mean       50%
is_canceled                        0.00     1.0     0.370416     0.000
lead_time                          0.00   737.0   104.011416    69.000
arrival_date_year               2015.00  2017.0  2016.156554  2016.000
arrival_date_week_number           1.00    53.0    27.165173    28.000
arrival_date_day_of_month          1.00    31.0    15.798241    16.000
stays_in_weekend_nights            0.00    19.0     0.927599     1.000
stays_in_week_nights               0.00    50.0     2.500302     2.000
adults                             0.00    55.0     1.856403     2.000
children                           0.00    10.0     0.103890     0.000
babies                             0.00    10.0     0.007949     0.000
is_repe

In [8]:
df_clean = df[(df['adr'] > 0) & (df['adr'] < 5000)].copy()

cols_drop = [
    'is_canceled', 'reservation_status', 'reservation_status_date',
    'agent', 'company', 'country', 'arrival_date_year'
]

df_model = df_clean.drop(columns=cols_drop, errors='ignore')

In [9]:
num_cols = df_model.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df_model.select_dtypes(include=['object', 'category']).columns.tolist()

pipe_num = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
    ('scaler', StandardScaler())
])

pipe_cat = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preproc = ColumnTransformer(
    transformers=[
        ('num', pipe_num, num_cols),
        ('cat', pipe_cat, cat_cols)
    ])

print("A processar os dados...")
X_final = preproc.fit_transform(df_model)
print(f"Dimensão dos dados processados: {X_final.shape}")

A processar os dados...
Dimensão dos dados processados: (117429, 75)


In [10]:
k_values = range(3, 7)
resultados = []

print("\n--- K-Means a serem Treinados ---")
for k in k_values:
    print(f"A treinar modelo com k={k}...")
    
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_final)
    
    sil = silhouette_score(X_final, labels, sample_size=30000, random_state=42)
    ch = calinski_harabasz_score(X_final, labels)
    
    resultados.append({
        'K': k,
        'Inércia': kmeans.inertia_,
        'Silhouette': sil,
        'Calinski-Harabasz': ch
    })

df_res = pd.DataFrame(resultados)
print("\n--- Resultados da Avaliação ---")
print(df_res.to_string(index=False))


--- K-Means a serem Treinados ---
A treinar modelo com k=3...
A treinar modelo com k=4...
A treinar modelo com k=5...
A treinar modelo com k=6...

--- Resultados da Avaliação ---
 K      Inércia  Silhouette  Calinski-Harabasz
 3 2.088047e+06    0.111345        8478.982480
 4 1.997246e+06    0.077109        7689.112472
 5 1.899269e+06    0.091669        7578.648992
 6 1.819334e+06    0.086083        7361.083402


In [11]:
print("\nA treinar K-Means final com K=3")

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_clean['Cluster'] = kmeans.fit_predict(X_final)


A treinar K-Means final com K=3


In [12]:
print("\n--- Perfil do Cluster ---")

vars_chave = ['lead_time', 'adr', 'stays_in_week_nights', 'stays_in_weekend_nights', 'adults', 'children']

perfil = df_clean.groupby('Cluster')[vars_chave].mean().round(2)
perfil['Total_Reservas'] = df_clean.groupby('Cluster').size()

print(perfil.to_string())


--- Perfil do Cluster ---
         lead_time     adr  stays_in_week_nights  stays_in_weekend_nights  adults  children  Total_Reservas
Cluster                                                                                                    
0            35.28   74.76                  1.61                     0.49    1.38      0.03            3332
1            90.69  134.95                  3.35                     1.35    2.10      0.27           42665
2           116.94   86.03                  2.07                     0.71    1.74      0.01           71432


In [13]:
print("\n--- Distribuição por Hotéis ---")
print(pd.crosstab(df_clean['Cluster'], df_clean['hotel'], normalize='index').round(2) * 100)


--- Distribuição por Hotéis ---
hotel    City Hotel  Resort Hotel
Cluster                          
0              50.0          50.0
1              55.0          45.0
2              74.0          26.0
